In [1]:
import pandas as pd
import requests

print("Notebook funktioniert!")
print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)

Notebook funktioniert!
Pandas: 3.0.6
Requests: 2.34.2


In [5]:
import requests
from datetime import datetime, timezone
from uuid import uuid4

URL = "https://api.opentransportdata.swiss/ojp20"

STOP_ID = "ch:1:sloid:3000"
STOP_NAME = "Zürich HB"

now_utc = datetime.now(timezone.utc)
timestamp = now_utc.isoformat(timespec="milliseconds").replace("+00:00", "Z")

message_id = f"zhaw-pilot-{uuid4()}"

xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xsi:schemaLocation="http://www.vdv.de/ojp"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:ServiceRequestContext>
                <siri:Language>de</siri:Language>
            </siri:ServiceRequestContext>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPStopEventRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <Location>

                    <PlaceRef>

                        <siri:StopPointRef>{STOP_ID}</siri:StopPointRef>

                        <Name>
                            <Text>{STOP_NAME}</Text>
                        </Name>

                    </PlaceRef>

                    <DepArrTime>{timestamp}</DepArrTime>

                </Location>

                <Params>

                    <NumberOfResults>10</NumberOfResults>

                    <StopEventType>departure</StopEventType>

                    <IncludePreviousCalls>false</IncludePreviousCalls>

                    <IncludeOnwardCalls>false</IncludeOnwardCalls>

                    <UseRealtimeData>full</UseRealtimeData>

                </Params>

            </OJPStopEventRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

headers = {
    "Content-Type": "application/xml",
    "Authorization": f"Bearer {TOKEN}"
}

response = requests.post(
    URL,
    headers=headers,
    data=xml_request.encode("utf-8"),
    timeout=30
)

print("HTTP Status:", response.status_code)
print()
print("Antwort der API:")
print(response.text[:3000])

HTTP Status: 200

Antwort der API:
<?xml version="1.0" encoding="utf-8"?><OJP xmlns:siri="http://www.siri.org.uk/siri" version="2.0" xmlns="http://www.vdv.de/ojp"><OJPResponse><siri:ServiceDelivery><siri:ResponseTimestamp>2026-09-19T11:50:37.8041852+02:00</siri:ResponseTimestamp><siri:ProducerRef>MENTZ-prod-asg_1.1.17.555_i-0d9ca8b0d9</siri:ProducerRef><siri:ResponseMessageIdentifier>00-199098173e6f5651f802b</siri:ResponseMessageIdentifier><OJPStopEventDelivery><siri:ResponseTimestamp>2026-09-19T11:50:37.8036273+02:00</siri:ResponseTimestamp><siri:RequestMessageRef>zhaw-pilot-bdb85df8-556d-4442-9263-2a6200c38b95</siri:RequestMessageRef><siri:DefaultLanguage>de</siri:DefaultLanguage><CalcTime>44</CalcTime><StopEventResponseContext><Places><Place><StopPlace><StopPlaceRef>ch:1:sloid:3000</StopPlaceRef><StopPlaceName><Text xml:lang="de">Zürich HB</Text></StopPlaceName><PrivateCode><System>EFA</System><Value>108276:0:31</Value></PrivateCode><TopographicPlaceRef>23026261:27</TopographicPlace

In [6]:
import xml.etree.ElementTree as ET

# XML-Antwort in eine durchsuchbare Struktur umwandeln
root = ET.fromstring(response.content)

# XML namespaces
ns = {
    "ojp": "http://www.vdv.de/ojp",
    "siri": "http://www.siri.org.uk/siri"
}

# Alle gefundenen Verbindungen / Stop Events suchen
results = root.findall(".//ojp:StopEventResult", ns)

print("Gefundene Verbindungen:", len(results))

# Erste Verbindung vollständig anzeigen
if len(results) > 0:
    first_result = ET.tostring(
        results[0],
        encoding="unicode"
    )

    print("\nErste Verbindung:")
    print(first_result[:5000])
else:
    print("Keine StopEventResult gefunden.")

Gefundene Verbindungen: 10

Erste Verbindung:
<ns0:StopEventResult xmlns:ns0="http://www.vdv.de/ojp" xmlns:ns1="http://www.siri.org.uk/siri"><ns0:Id>268fd067-9098-40f0-bacb-7de44784641c</ns0:Id><ns0:StopEvent><ns0:ThisCall><ns0:CallAtStop><ns1:StopPointRef>ch:1:sloid:3000:500:31</ns1:StopPointRef><ns0:StopPointName><ns0:Text xml:lang="de">Zürich HB</ns0:Text></ns0:StopPointName><ns0:NameSuffix><ns0:Text xml:lang="de">PLATFORM_ACCESS_WITHOUT_ASSISTANCE</ns0:Text></ns0:NameSuffix><ns0:PlannedQuay><ns0:Text xml:lang="de">31</ns0:Text></ns0:PlannedQuay><ns0:ServiceDeparture><ns0:TimetabledTime>2026-09-19T09:49:00Z</ns0:TimetabledTime><ns0:EstimatedTime>2026-09-19T09:50:06Z</ns0:EstimatedTime></ns0:ServiceDeparture><ns0:Order>10</ns0:Order><ns1:ExpectedDepartureOccupancy><ns1:FareClass>firstClass</ns1:FareClass><ns1:OccupancyLevel>manySeatsAvailable</ns1:OccupancyLevel></ns1:ExpectedDepartureOccupancy><ns1:ExpectedDepartureOccupancy><ns1:FareClass>secondClass </ns1:FareClass><ns1:OccupancyL

In [8]:
import pandas as pd
import xml.etree.ElementTree as ET

# XML erneut parsen
root = ET.fromstring(response.content)

# Namespaces
ns = {
    "ojp": "http://www.vdv.de/ojp",
    "siri": "http://www.siri.org.uk/siri"
}

# Alle StopEventResult-Elemente finden
results = root.findall(".//ojp:StopEventResult", ns)

rows = []

for result in results:

    stop_event = result.find("ojp:StopEvent", ns)

    if stop_event is None:
        continue

    # --------------------------------------------------
    # 1. Informationen zum aktuellen Halt
    # --------------------------------------------------

    this_call = stop_event.find(
        "ojp:ThisCall/ojp:CallAtStop",
        ns
    )

    if this_call is None:
        continue

    stop_point_ref = this_call.findtext(
        "siri:StopPointRef",
        default=None,
        namespaces=ns
    )

    stop_name = this_call.findtext(
        "ojp:StopPointName/ojp:Text",
        default=None,
        namespaces=ns
    )

    planned_platform = this_call.findtext(
        "ojp:PlannedQuay/ojp:Text",
        default=None,
        namespaces=ns
    )

    scheduled_departure = this_call.findtext(
        "ojp:ServiceDeparture/ojp:TimetabledTime",
        default=None,
        namespaces=ns
    )

    estimated_departure = this_call.findtext(
        "ojp:ServiceDeparture/ojp:EstimatedTime",
        default=None,
        namespaces=ns
    )

    # --------------------------------------------------
    # 2. Informationen zur Fahrt
    # --------------------------------------------------

    service = stop_event.find(
        "ojp:Service",
        ns
    )

    if service is None:
        continue

    operating_day = service.findtext(
        "ojp:OperatingDayRef",
        default=None,
        namespaces=ns
    )

    journey_ref = service.findtext(
        "ojp:JourneyRef",
        default=None,
        namespaces=ns
    )

    public_code = service.findtext(
        "ojp:PublicCode",
        default=None,
        namespaces=ns
    )

    transport_mode = service.findtext(
        "ojp:Mode/ojp:PtMode",
        default=None,
        namespaces=ns
    )

    product_category = service.findtext(
        "ojp:ProductCategory/ojp:Name/ojp:Text",
        default=None,
        namespaces=ns
    )

    line = service.findtext(
        "ojp:PublishedServiceName/ojp:Text",
        default=None,
        namespaces=ns
    )

    train_number = service.findtext(
        "ojp:TrainNumber",
        default=None,
        namespaces=ns
    )

    origin = service.findtext(
        "ojp:OriginText/ojp:Text",
        default=None,
        namespaces=ns
    )

    destination = service.findtext(
        "ojp:DestinationText/ojp:Text",
        default=None,
        namespaces=ns
    )

    # --------------------------------------------------
    # 3. Als Zeile speichern
    # --------------------------------------------------

    rows.append({
        "operating_day": operating_day,
        "journey_ref": journey_ref,
        "stop_point_ref": stop_point_ref,
        "station_name": stop_name,
        "transport_mode": transport_mode,
        "product_category": product_category,
        "public_code": public_code,
        "line": line,
        "train_number": train_number,
        "origin": origin,
        "destination": destination,
        "planned_platform": planned_platform,
        "scheduled_departure": scheduled_departure,
        "estimated_departure": estimated_departure
    })


# ============================================================
# DATAFRAME ERSTELLEN
# ============================================================

df = pd.DataFrame(rows)

print("Anzahl Verbindungen:", len(df))

display(df)

# Zeitspalten in echte datetime-Werte umwandeln

df["scheduled_departure"] = pd.to_datetime(
    df["scheduled_departure"],
    utc=True,
    errors="coerce"
)

df["estimated_departure"] = pd.to_datetime(
    df["estimated_departure"],
    utc=True,
    errors="coerce"
)


# Schweizer Lokalzeit erzeugen

df["scheduled_departure_local"] = (
    df["scheduled_departure"]
    .dt.tz_convert("Europe/Zurich")
)

df["estimated_departure_local"] = (
    df["estimated_departure"]
    .dt.tz_convert("Europe/Zurich")
)


# Prüfen, ob Echtzeitinformation vorhanden ist

df["has_realtime"] = (
    df["estimated_departure"]
    .notna()
)


# Verspätung berechnen

df["predicted_delay_minutes"] = (
    df["estimated_departure"]
    - df["scheduled_departure"]
).dt.total_seconds() / 60


# Wichtige Spalten anzeigen

display(
    df[
        [
            "station_name",
            "transport_mode",
            "product_category",
            "line",
            "origin",
            "destination",
            "planned_platform",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ]
)

Anzahl Verbindungen: 10


,operating_day,journey_ref,stop_point_ref,station_name,transport_mode,product_category,public_code,line,train_number,origin,destination,planned_platform,scheduled_departure,estimated_departure
0,2026-09-19,ch:1:sjyid:100001:19442-001,ch:1:sloid:3000:500:31,Zürich HB,rail,S-Bahn,S14,S14,19442,Hinwil,Affoltern am Albis,31,2026-09-19T09:49:00Z,2026-09-19T09:50:06Z
1,2026-09-19,ch:1:sjyid:100001:20443-001,ch:1:sloid:3000:3:4,Zürich HB,rail,S-Bahn,S24,S24,20443,Effretikon,Zug,4,2026-09-19T09:51:00Z,2026-09-19T09:51:18Z
2,2026-09-19,ch:1:sjyid:100001:2067-001,ch:1:sloid:3000:501:33,Zürich HB,rail,InterRegio,IR36,IR36,2067,Basel SBB,Zürich Flughafen,33,2026-09-19T09:52:00Z,2026-09-19T09:52:24Z
3,2026-09-19,ch:1:sjyid:100001:19542-001,ch:1:sloid:3000:502:42,Zürich HB,rail,S-Bahn,S15,S15,19542,Rapperswil SG,Niederweningen,41/42,2026-09-19T09:52:00Z,2026-09-19T09:52:24Z
4,2026-09-19,ch:1:sjyid:100001:2366-002,ch:1:sloid:3000:9:17,Zürich HB,rail,InterRegio,IR35,IR35,2366,Chur,Bern,17,2026-09-19T09:53:00Z,2026-09-19T09:53:30Z
5,2026-09-19,ch:1:sjyid:100001:18543-001,ch:1:sloid:3000:503:43,Zürich HB,rail,S-Bahn,S5,S5,18543,Zug,Pfäffikon SZ,43/44,2026-09-19T09:54:00Z,2026-09-19T09:54:18Z
6,2026-09-19,ch:1:sjyid:100001:18842-001,ch:1:sloid:3000:501:34,Zürich HB,rail,S-Bahn,S8,S8,18842,Pfäffikon SZ,Effretikon,34,2026-09-19T09:55:00Z,2026-09-19T09:55:18Z
7,2026-09-19,ch:1:sjyid:100001:18943-001,ch:1:sloid:3000:503:43,Zürich HB,rail,S-Bahn,S9,S9,18943,Schaffhausen,Uster,43/44,2026-09-19T09:58:00Z,2026-09-19T09:58:18Z
8,2026-09-19,ch:1:sjyid:100001:770-001,ch:1:sloid:3000:8:14,Zürich HB,rail,InterCity,IC3,IC3,770,Chur,Basel SBB,14,2026-09-19T09:59:00Z,2026-09-19T09:59:30Z
9,2026-09-19,ch:1:sjyid:100001:19144-001,ch:1:sloid:3000:502:42,Zürich HB,rail,S-Bahn,S11,S11,19144,Zürich HB,Aarau,41/42,2026-09-19T09:59:00Z,2026-09-19T10:00:06Z


,station_name,transport_mode,product_category,line,origin,destination,planned_platform,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,Zürich HB,rail,S-Bahn,S14,Hinwil,Affoltern am Albis,31,2026-09-19 11:49:00+02:00,2026-09-19 11:50:06+02:00,1.1
1,Zürich HB,rail,S-Bahn,S24,Effretikon,Zug,4,2026-09-19 11:51:00+02:00,2026-09-19 11:51:18+02:00,0.3
2,Zürich HB,rail,InterRegio,IR36,Basel SBB,Zürich Flughafen,33,2026-09-19 11:52:00+02:00,2026-09-19 11:52:24+02:00,0.4
3,Zürich HB,rail,S-Bahn,S15,Rapperswil SG,Niederweningen,41/42,2026-09-19 11:52:00+02:00,2026-09-19 11:52:24+02:00,0.4
4,Zürich HB,rail,InterRegio,IR35,Chur,Bern,17,2026-09-19 11:53:00+02:00,2026-09-19 11:53:30+02:00,0.5
5,Zürich HB,rail,S-Bahn,S5,Zug,Pfäffikon SZ,43/44,2026-09-19 11:54:00+02:00,2026-09-19 11:54:18+02:00,0.3
6,Zürich HB,rail,S-Bahn,S8,Pfäffikon SZ,Effretikon,34,2026-09-19 11:55:00+02:00,2026-09-19 11:55:18+02:00,0.3
7,Zürich HB,rail,S-Bahn,S9,Schaffhausen,Uster,43/44,2026-09-19 11:58:00+02:00,2026-09-19 11:58:18+02:00,0.3
8,Zürich HB,rail,InterCity,IC3,Chur,Basel SBB,14,2026-09-19 11:59:00+02:00,2026-09-19 11:59:30+02:00,0.5
9,Zürich HB,rail,S-Bahn,S11,Zürich HB,Aarau,41/42,2026-09-19 11:59:00+02:00,2026-09-19 12:00:06+02:00,1.1
